# A 25 m terrain and geological map for GeoModeller

Default example: **Cervin / Matterhorn, 6 x 6 km, north-up**. This notebook downloads swissALTI3D, averages it to 25 m, downloads the geological map from map.geo.admin.ch, and exports an ASCII terrain plus matching shaded relief and geology PNGs.

**Edit only the values below, then run all cells from top to bottom.** Use the **Python (swiss-dem)** kernel. On NOTO, calculations and files stay on the server until you download the final ZIP. See the README for setup and GeoModeller import.

## 1. EDIT HERE — study area and geology

Keep these values for the Cervin test. For another area:

- **`CENTER_LV95`**: centre of the square, `(x, y)` in LV95 metres.
- **`SIDE_M`**: side length, between 100 m and 10 km. Use `(width, height)` for a rectangle. Sides are rounded up to whole 25 m cells.
- **`AZIMUTH_DEG`**: direction the **top edge** of the square faces, in degrees clockwise from north. `0` = north-up; `30` = the square is turned 30° clockwise, so its top edge faces N30°E. Use this to align the model with a structure, such as a fold axis or a cross-section line.
- **`GEOLOGY`**: `"geocover"` (GeoCover, clean vector map), `"atlas"` (scanned Geological Atlas 1:25 000, with symbols and labels), or `None` for no geological map.

**A rotated area (azimuth not 0) is exported in local coordinates**, in metres from its lower-left corner, because GeoModeller grids and project limits are aligned with the X/Y axes. `project_info.txt` gives the LV95 corners and the conversion formulas.

In [ ]:
# ONLY CHANGE THESE VALUES — default: Cervin / Matterhorn, 6 x 6 km, north-up.
CENTER_LV95 = (2_617_000, 1_092_000)  # centre (x, y) in LV95 metres
SIDE_M = 6_000                        # side length in metres, or (width, height)
AZIMUTH_DEG = 0                       # direction the top edge faces, clockwise from north
GEOLOGY = "geocover"                  # "geocover", "atlas" or None

## 2. Setup and validate — run unchanged

Locate the project and validate the area before downloading. Implementation helpers live in `dem_tools.py`; keep that file alongside the README when copying this project.

In [ ]:
from pathlib import Path
import sys

# Works from the project root or notebooks/, including a GitHub ZIP folder.
PROJECT_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "dem_tools.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook inside the complete get_dem project folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dem_tools import (make_area, find_tiles, download_tiles, get_geology,
                       create_grid, export_outputs, preview)

area = make_area(CENTER_LV95, SIDE_M, AZIMUTH_DEG)
CACHE = PROJECT_ROOT / "data" / "swissalti3d_tiles"
OUTPUT = PROJECT_ROOT / "output"
print(f"Area: {area.width/1000:g} x {area.height/1000:g} km, top edge facing {area.azimuth:g}°")
print("Coordinates in the exported files:", "LV95" if area.north_up else "local metres (rotated area)")
print(f"Python: {sys.executable}")

## 3. Download the terrain source — run unchanged

Internet access is required for the swisstopo catalog and uncached tiles. Only 2 m LV95/LN02 tiles are selected, using the latest available edition of each tile. A rotated area needs every tile under its north-up envelope, so it downloads more tiles than a north-up one of the same size. Downloads are checked before being reused. Source TIFFs stay in the cache; GeoModeller does not need them.

In [ ]:
tiles = find_tiles(area)
print(f"Source tiles: {len(tiles)}")
tile_paths = download_tiles(tiles, CACHE)

## 4. Download the geological map — run unchanged

The map is requested from the map.geo.admin.ch WMS at 5 m pixels, then turned to match the area. It has the same outer edges as the terrain. Transparent pixels have no mapped geology. To read the legend, open the map.geo.admin.ch link written in `project_info.txt` and click on the map.

In [ ]:
geology = get_geology(area, GEOLOGY)
print("Geology:", "not requested" if geology is None else geology["name"])

## 5. Build the terrain and export — run unchanged

Average the 2 m data once onto 25 m cells. Missing terrain is kept as NoData, not filled or interpolated across borders. The hillshade is calculated from the terrain without another download.

This step replaces the named files in `output`. The ZIP contains only this run's files; download or rename a previous result before changing the area.

In [ ]:
dem, transform = create_grid(tile_paths, area)
info, rgba, archive = export_outputs(dem, area, OUTPUT, tiles, geology)
print(f"Grid: {dem.shape[1]} columns x {dem.shape[0]} rows at 25 m")
print(f"Valid terrain coverage: {info['coverage_percent']:.1f}%")
if info['coverage_percent'] < 100:
    print("Some cells have no data (for example beyond Swiss coverage). They remain empty.")
print(f"Download this file after checking the preview: {archive}")

## 6. Project limits for GeoModeller — copy all six

For a **north-up** area, select **CH1903+ / LV95 (EPSG:2056)** and metre units in GeoModeller. For a **rotated** area, the limits below are local metres starting at 0; do not select LV95. Do not keep a project extent of 0–1000 unless it matches. ZMin/ZMax surround the terrain with at least 100 m margin; lower ZMin if your geological model should extend deeper.

The same values and the image registration coordinates are saved in `project_info.txt` inside the ZIP. For the default Cervin example, expect a **240 x 240** grid.

In [ ]:
for axis in "XYZ":
    print(f"{axis}Min = {info[axis+'Min']:.2f}    {axis}Max = {info[axis+'Max']:.2f}")
print(f"Terrain elevations (LN02): {info['elevation_min']:.2f} to {info['elevation_max']:.2f} m")

## 7. Check and download

The top-left panel shows the area in LV95; the red line is its top edge. The other panels use the exported coordinates, so their top is the area's top edge. Lighting is from the northwest, at 45 degrees, whatever the rotation; gaps are transparent.

After checking the preview, download **`output/geomodeller_dem.zip`** from NOTO's file browser and extract it on the GeoModeller computer. Import the `.asc` as **Digital Terrain Model / Arc Grid**. Add the PNGs through **Image Manager** on `SurfaceTopography`. Use `project_info.txt` for the project limits and image positioning. Follow README steps 5 and 6.

In [ ]:
preview(dem, rgba, area, geology)